# N型主升浪交易系统 — 交互式分析

本 Notebook 用于交互式探索 N 型结构信号，包括：
- 获取行情数据
- 计算技术指标
- N 型结构识别
- 信号评分
- 回测分析
- 图表可视化

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
from config import Config, get_aggressive_config, get_conservative_config
from data_utils import fetch_stock_hist, add_all_indicators, get_stock_list
from n_pattern import NPatternDetector, detect_all_signals, get_latest_signal, compute_signal_score
from backtest import BacktestEngine
from screener import StockScreener, quick_screen

# 设置 pandas 显示选项
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

print('✅ 模块导入成功')

## 1. 配置参数

选择预设配置或自定义参数。

In [ ]:
# 三选一
cfg = Config()                        # 默认配置
# cfg = get_aggressive_config()        # 激进配置（更多信号）
# cfg = get_conservative_config()      # 保守配置（更高胜率）

# 可选：微调参数
# cfg.n_pattern.fib_min = 0.35
# cfg.screener.min_score = 60
# cfg.risk.risk_percent = 1.5

print(f'均线: EMA{cfg.ma.ema_short}/{cfg.ma.ema_mid}/{cfg.ma.ema_long}')
print(f'回调区间: {cfg.n_pattern.fib_min*100:.0f}%-{cfg.n_pattern.fib_max*100:.0f}%')
print(f'单笔风险: {cfg.risk.risk_percent}%')
print(f'止盈RR: {cfg.risk.tp_rr_1}/{cfg.risk.tp_rr_2}')

## 2. 分析单只股票

In [ ]:
# 输入股票代码和周期
SYMBOL = '600519'   # 贵州茅台
PERIOD = 'weekly'   # weekly / daily / monthly

# 获取数据
df = fetch_stock_hist(SYMBOL, period=PERIOD, cache_dir=cfg.data.cache_dir)
print(f'{SYMBOL}: {len(df)} 根K线')
print(f'时间范围: {df.index[0]} ~ {df.index[-1]}')
df.tail(5)

In [ ]:
# 计算指标
df = add_all_indicators(df, cfg)

# 检测N型信号
df = detect_all_signals(df, cfg)

# 查看有信号的列
signal_cols = ['close', 'n_l1', 'n_h1', 'n_l2', 'n_ready', 'n_phase',
               'n_breakout', 'signal_score', 'signal_grade', 'buy_signal']
df[signal_cols].tail(10)

In [ ]:
# 最新信号详情
signal = get_latest_signal(df, cfg)

print(f'=== {SYMBOL} 最新信号 ===')
print(f'日期:    {str(signal.date)[:10]}')
print(f'评分:    {signal.score} 分 [{signal.grade}级]')
print(f'阶段:    {signal.n_pattern.phase}')
print(f'买入信号: {"🔥 是" if signal.buy_signal else "否"}')
print()
print(f'N型结构:')
print(f'  L1: {signal.n_pattern.l1_price:.2f}' if not np.isnan(signal.n_pattern.l1_price) else '  L1: --')
print(f'  H1: {signal.n_pattern.h1_price:.2f}' if not np.isnan(signal.n_pattern.h1_price) else '  H1: --')
print(f'  L2: {signal.n_pattern.l2_price:.2f}' if not np.isnan(signal.n_pattern.l2_price) else '  L2: --')
if signal.n_pattern.ready:
    print(f'  首波涨幅: {signal.n_pattern.leg_pct:.1f}%')
    print(f'  回调幅度: {signal.n_pattern.retrace_pct:.1f}%')
print()
print(f'条件检查:')
print(f'  均线: {"✅" if signal.ma_bullish else "❌"}')
print(f'  RSI:  {"✅" if signal.rsi_healthy else "❌"}')
print(f'  量能: {"✅" if signal.vol_burst else "❌"}')
print(f'  MACD: {"✅" if signal.macd_bullish else "❌"}')
print(f'  ADX:  {"✅" if signal.adx_trend else "❌"}')
if signal.buy_signal:
    print()
    print(f'交易计划:')
    print(f'  入场: {signal.entry_price:.2f}')
    print(f'  止损: {signal.stop_loss:.2f}')
    print(f'  TP1:  {signal.tp1:.2f}')
    print(f'  TP2:  {signal.tp2:.2f}')

## 3. 图表可视化

In [ ]:
# 简单版（只需matplotlib）
from visualize import plot_simple

plot_simple(df.tail(200), title=f'{SYMBOL} N型主升浪分析')

In [ ]:
# 完整版（需要 mplfinance）
# from visualize import plot_analysis
# plot_analysis(df.tail(150), cfg, title=f'{SYMBOL} N型主升浪分析')

## 4. 回测

In [ ]:
engine = BacktestEngine(cfg)
result = engine.run(df, symbol=SYMBOL)

print(engine.summary(result))

# 权益曲线
print(f'\n初始资金: ¥{cfg.backtest.initial_capital:,.0f}')
print(f'最终权益: ¥{result.equity_curve.iloc[-1]:,.0f}')
print(f'最大回撤: {result.max_drawdown:.1f}%')

In [ ]:
# 查看具体交易
if result.trades:
    trades_df = pd.DataFrame([{
        '入场': str(t.entry_date)[:10],
        '出场': str(t.exit_date)[:10],
        '入场价': t.entry_price,
        '出场价': t.exit_price,
        '盈亏': f'{t.pnl:+,.0f}',
        '盈亏%': f'{t.pnl_pct:+.1f}%',
        '原因': t.exit_reason,
        '持仓': t.holding_bars,
        '评分': t.score,
        '等级': t.grade,
    } for t in result.trades])
    display(trades_df)
else:
    print('无交易记录')

## 5. 批量选股

In [ ]:
# 快速筛选自定义列表
symbols = ['000001', '000002', '000858', '600519', '601318', '600036']
results = quick_screen(symbols, cfg)

if not results.empty:
    display(results[['code', 'close', 'score', 'grade', 'phase', 'leg_pct', 'retrace_pct', 'buy_signal']])
else:
    print('未找到符合条件的品种')

In [ ]:
# 全市场筛选（耗时较长，谨慎运行）
# screener = StockScreener(cfg)
# cfg.screener.min_score = 65
# cfg.screener.top_n = 30
# results = screener.run()
# screener.print_results(results)

## 6. 参数探索

比较不同斐波那契回调区间对信号数量的影响。

In [ ]:
# 参数敏感性分析
fib_ranges = [
    (0.25, 0.70, '宽松'),
    (0.30, 0.65, '默认'),
    (0.35, 0.60, '中等'),
    (0.382, 0.55, '严格'),
]

for fib_min, fib_max, label in fib_ranges:
    test_cfg = Config()
    test_cfg.n_pattern.fib_min = fib_min
    test_cfg.n_pattern.fib_max = fib_max

    df_test = add_all_indicators(df.copy(), test_cfg)
    df_test = detect_all_signals(df_test, test_cfg)

    buy_count = df_test['buy_signal'].sum()
    ready_count = df_test['n_ready'].sum()
    print(f'{label} [{fib_min*100:.0f}%-{fib_max*100:.0f}%]: '
          f'结构就绪 {ready_count}次, 买入信号 {buy_count}次')